# 02 — Abstract Base Classes and Duck Typing

How do you say "every subclass MUST implement `area`"? With an **abstract base class** (ABC). And how does Python decide if an object "counts" as a Shape without inheritance? **Duck typing.**

We touch both here, and contrast them with `Protocol` (folder 05).

## The problem

Without an ABC, a forgetful subclass breaks at the worst time.

In [1]:
class Shape:
    def area(self):
        # No body — what should this do? raise? return 0? Nothing is enforced.
        raise NotImplementedError

class Triangle(Shape):
    def __init__(self, base, height):
        self.base = base
        self.height = height
    # Oops — forgot to implement area.

t = Triangle(3, 4)
# Triangle is happily *constructed*. The bug surfaces only when someone calls area():
try:
    print(t.area())
except NotImplementedError:
    print("Construction succeeded; area() blew up. Late failure.")

Construction succeeded; area() blew up. Late failure.


## `abc.ABC` + `@abstractmethod` — fail at construction time

In [2]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @abstractmethod
    def area(self) -> float: ...

    @abstractmethod
    def perimeter(self) -> float: ...

    # Non-abstract methods are fine — they're inherited as usual:
    def describe(self) -> str:
        return f"{type(self).__name__}: area={self.area():.2f}, perimeter={self.perimeter():.2f}"

# Now we cannot even construct Shape itself:
try:
    Shape()
except TypeError as e:
    print("rejected:", e)

# And we cannot construct a subclass that forgot a required method:
class BadTriangle(Shape):
    def __init__(self, base, height): self.base, self.height = base, height
    def area(self): return 0.5 * self.base * self.height
    # forgot perimeter

try:
    BadTriangle(3, 4)
except TypeError as e:
    print("rejected at construction:", e)

rejected: Can't instantiate abstract class Shape without an implementation for abstract methods 'area', 'perimeter'
rejected at construction: Can't instantiate abstract class BadTriangle without an implementation for abstract method 'perimeter'


### A complete subclass works fine

In [3]:
import math

class Circle(Shape):
    def __init__(self, radius: float):
        self.radius = radius
    def area(self) -> float:
        return math.pi * self.radius ** 2
    def perimeter(self) -> float:
        return 2 * math.pi * self.radius

class Rectangle(Shape):
    def __init__(self, w: float, h: float):
        self.w, self.h = w, h
    def area(self) -> float:
        return self.w * self.h
    def perimeter(self) -> float:
        return 2 * (self.w + self.h)

for s in [Circle(3), Rectangle(2, 4)]:
    print(s.describe())   # inherited helper that USES the polymorphic methods

Circle: area=28.27, perimeter=18.85
Rectangle: area=8.00, perimeter=12.00


## Why this matters for what's next

- **Pydantic v2** (`BaseModel`) is an ABC-like contract — subclasses *must* declare fields.
- **FastAPI** dependencies are often defined as a contract ("things that produce a `User`") with multiple swappable implementations — exactly this pattern.
- **Agent frameworks** define an `LLMClient` interface and let you plug in OpenAI, Anthropic, or a mock — same shape.

Treat ABCs as *the way to write down a contract*. The next folder shows another way (`Protocol`) that doesn't even require inheritance.

## Duck typing — the Python way

*If it walks like a duck and quacks like a duck, it's a duck.*

Python doesn't require an object to inherit from `Shape` for `s.area()` to work — it only requires that `s` *has* an `area` method. This is **duck typing**: behavior matters, declared type doesn't.

In [4]:
class FakeShape:
    # Doesn't inherit from Shape. Doesn't claim to be one.
    def area(self): return 999
    def perimeter(self): return 0

def total_area(items):
    return sum(s.area() for s in items)

# Mixed in with a real Shape subclass — works fine:
print(total_area([Circle(2), Rectangle(3, 4), FakeShape()]))

1023.5663706143591


### Trade-offs

| | ABC | Duck typing |
|---|---|---|
| **Enforces contract** | yes (at construction) | no (fails late, if at all) |
| **Requires inheritance** | yes | no |
| **Easy to test with mocks** | medium | easy (any object with the right methods works) |
| **IDE autocomplete / type checker** | great | okay if you type-hint with `Protocol` |

**The pragmatic answer**, which we'll formalize in folder 05: use **`Protocol`** to declare
the contract for type checking, while still allowing duck-typed implementations. Best of both.

## `isinstance` with an ABC

`isinstance(x, Shape)` is `True` for real subclasses, `False` for `FakeShape` even though it has the methods. This is the price of ABCs — and the reason `Protocol` exists.

In [5]:
print(isinstance(Circle(1), Shape))    # True
print(isinstance(FakeShape(), Shape))  # False — same methods, but not a Shape subclass

True
False


## Mini-exercise

1. Add `Triangle(base, height)` to the `Shape` hierarchy. Implement both abstract methods.
   For perimeter assume an isoceles triangle (`base + 2 * slant`, compute slant from base/height).
2. Write `print_largest(shapes)` that prints the largest shape by area. Use polymorphism —
   *no `isinstance`*.
3. Make `Shape` define `__repr__` to print `<ClassName area=...>`. Verify it shows the
   right class name for `Circle`, `Rectangle`, `Triangle`. How does `type(self).__name__`
   make this trivial?

In [6]:
# your solutions here


**Takeaways**

- `abc.ABC` + `@abstractmethod` enforces "every subclass must implement this" at *construction* time.
- Concrete (non-abstract) methods on the ABC can call abstract ones — that's the **template method** pattern.
- Duck typing works without inheritance, but skips the contract check. We get both back with `Protocol` in folder 05.

Next: [03_dataclasses_and_property.ipynb](03_dataclasses_and_property.ipynb) — skip the boilerplate.